In [2]:
import folium
import pandas as pd

# Read data from CSV file
df = pd.read_csv('locations.csv')

# Create separate dataframes for each type based on name prefix
vendors_data = df[df['name'].str.startswith('vendor')].to_dict('records')
stores_data = df[df['name'].str.startswith('store')].to_dict('records')
dc_data = df[df['name'].str.startswith('dc')].to_dict('records')

print(dc_data)

# Create a map centered on US
m = folium.Map(location=[39.8283, -98.5795], zoom_start=4)


# Add vendors (red markers)
for vendor in vendors_data:
    folium.Marker(
        location=[vendor['latitude'], vendor['longitude']],
        popup=vendor['name'],
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(m)

# Add stores (blue markers)
for store in stores_data:
    folium.Marker(
        location=[store['latitude'], store['longitude']],
        popup=store['name'],
        icon=folium.Icon(color='blue', icon='info-sign'),
    ).add_to(m)

# Add DCs (green markers)
for dc in dc_data:
    folium.Marker(
        location=[dc['latitude'], dc['longitude']],
        popup=dc['name'],
        icon=folium.Icon(color='green', icon='info-sign'),
    ).add_to(m)

# Add a legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 150px; height: 90px; 
            border:2px solid grey; z-index:9999; background-color:white;
            opacity:0.8;
            padding: 10px;
            font-size:14px;">
    <p>
        <i class="fa fa-circle fa-1x" style="color:red"></i> Vendors<br>
        <i class="fa fa-circle fa-1x" style="color:blue"></i> Stores<br>
        <i class="fa fa-circle fa-1x" style="color:green"></i> DCs
    </p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save the map
m.save('us_locations_map.html')

[{'name': 'dc0', 'latitude': 39.30473399445198, 'longitude': -85.8221778887178}, {'name': 'dc1', 'latitude': 40.51606620847434, 'longitude': -108.78148481155796}]


In [ ]:
import folium
from typing import Optional

def visualize_network(network: Network, save_path: Optional[str] = "network_map.html"):
    """
    Creates an interactive map visualization of the network nodes and edges.
    
    Parameters:
    - network: Network object containing nodes and edges
    - save_path: Path to save the HTML map file (default: "network_map.html")
    """
    # Create a map centered on the mean coordinates of all nodes
    center_lat = sum(node.lat for node in network.nodes.values()) / len(network.nodes)
    center_lon = sum(node.lon for node in network.nodes.values()) / len(network.nodes)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=6)
    
    # Add nodes to the map
    for node_id, node in network.nodes.items():
        color = 'red' if node_id in network.dcs else 'blue'
        tooltip = f"Node {node_id}"
        if node_id in network.dcs:
            tooltip += " (DC)"
            
        folium.CircleMarker(
            location=[node.lat, node.lon],
            radius=6,
            color=color,
            fill=True,
            popup=tooltip,
            tooltip=tooltip
        ).add_to(m)
    
    # Add edges to the map
    for edge_id, edge in network.edges.items():
        start_node = network.nodes[edge.start]
        end_node = network.nodes[edge.end]
        
        # Create a line for the edge
        points = [[start_node.lat, start_node.lon], [end_node.lat, end_node.lon]]
        tooltip = f"Edge {edge_id}: {edge.start} → {edge.end}\nDemand: {edge.demand}"
        
        folium.PolyLine(
            points,
            weight=2,
            color='gray',
            opacity=0.8,
            tooltip=tooltip
        ).add_to(m)
    
    # Save the map
    m.save(save_path)
    print(f"Map saved to {save_path}")
